<a href="https://colab.research.google.com/github/Wennylai0208/ZK-VCG/blob/main/Qimei_Island_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 基於 VCG 拍賣機制的孤島微電網快速頻率響應
## —— 以七美島為案例

**作者：** Wenny
**日期：** 2026 年 4 月
**關鍵字：** VCG 拍賣、快速頻率響應、孤島微電網、七美島、頻率控制

## 摘要

孤島微電網在高再生能源滲透下面臨秒級頻率危機。近年微電網與 P2P 能源交易研究沿三股趨勢推進：**機制設計追求激勵相容**（Long et al., 2018）、**區塊鏈支持去中心化**（Kang et al., 2017）、**密碼學保護隱私**（Guan et al., 2018）。然而三者各自引入新的限制，導致在秒級控制的實務場景中仍存在明顯空缺：VCG 缺隱私、區塊鏈 Gas 成本過高、ZK 證明生成需 10–30 秒。

本研究應用 VCG（Vickrey–Clarke–Groves）拍賣機制，搭配預計算策略（Radovanovic et al., 2023），設計能在秒級內運作的快速頻率響應控制。當頻率下跌時自動公平選擇停機的可控負載（比特幣礦機），平衡功率，避免系統觸發低頻保護（UFLS）。

以台灣七美島微電網為案例進行模擬驗證，結果顯示：無控制時頻率下跌至 57.13 Hz 觸發黑停；採用 VCG 控制後頻率維持在 57.98 Hz，成功避免黑停。本研究證明 VCG 機制能有效解決孤島微電網的秒級頻率穩定問題，並指出 ZK-VCG 作為未來隱私保護的擴展路徑。

## 第一章　研究背景與動機

### 1.1 研究背景

近年來，再生能源在電力系統中佔比快速提升，特別是太陽能發電。然而太陽能的高變動性對電網穩定造成挑戰，尤其在離島型微電網中尤為明顯。台灣七美島是典型案例：島上有約 800 kW 太陽能（355 kWp 裝置容量）、600 kW 柴油機，晴天時太陽能佔主要供電。但當雲朵遮蔽太陽時，太陽能在**秒級**內快速下降，導致功率失衡，系統頻率急劇下跌。

與此同時，微電網與 P2P 能源交易研究沿三股趨勢推進：

| 趨勢 | 代表研究 | 優勢 | 核心限制 |
|------|----------|------|----------|
| **機制設計：激勵相容** | Long et al. (2018) | VCG 保證誠實出價為納什均衡，社會福利最大化 | 依賴中央計算，缺乏隱私保護 |
| **區塊鏈：去中心化** | Kang et al. (2017) | 去中心化執行、透明可審計 | 出價公開洩露隱私；Gas 成本數百美元，難以頻繁執行 |
| **密碼學：隱私保護** | Guan et al. (2018) | ZK 證明可驗證計算而不洩露出價 | ZK 證明生成需 10–30 秒，秒級控制延遲超標 |

三股趨勢各自突破一部分問題，卻同時引入新的限制，導致在**秒級控制**的實務場景中仍存在明顯空缺。Radovanovic et al. (2023) 在碳感知資料中心研究中提出「**決策可慢、控制須快（< 1 秒）**」原則，以預計算策略拆分時間路徑——此觀念是本研究架構的核心依據。

### 1.2 問題陳述

七美島面臨以下挑戰：

**(1) 鴨子曲線現象：** 白天太陽能充足，傍晚需求增加但太陽能減少，造成劇烈的功率變化。依望安島縮放數據，七美島冬季傍晚 17–19 時淨負載從 856 kW 跳升至 1200 kW，兩小時爬升 344 kW，是電網最大壓力點。

**(2) 慣性低：** 孤島系統慣性常數 H ≈ 2.0 秒（依七美島 600 kW 柴油機組估算，Kundur, 2004），遠低於大型互聯電網（H = 5–10 秒），對擾動更敏感。

**(3) 控制時間窗口短：** 從擾動發生到觸發 UFLS 僅數秒，需要秒級反應（IEEE 1547-2018）。

**(4) 三難困境：** 激勵相容（VCG）、去中心化（區塊鏈）、隱私保護（ZK）三者目前無法同時在秒級場景中落地。

### 1.3 研究目的

本研究聚焦於**最小可行方案**：以 VCG 解決激勵相容與社會福利最大化，以**預計算策略**將報價排序離線完成，在頻率觸發點毫秒級執行停機決策，**隱私保護（ZK-VCG）列為未來擴展方向**。

具體目標：(1) 建立七美島微電網動態模擬模型；(2) 設計基於 VCG 的礦機停機決策算法；(3) 模擬驗證 VCG 機制控制效果；(4) 比較有無控制下系統頻率穩定性。

### 1.4 研究貢獻

1. 填補 Tushar (2020) 指出的研究缺口：首次將 VCG 機制應用於孤島微電網**秒級**快速頻率響應
2. 證明比特幣礦機可作為有效的快速可控負載（毫秒級停機響應）
3. 以預計算策略解決 VCG 的計算時間瓶頸，架構可推廣至 ZK-VCG
4. 為七美島及其他離島微電網提供具體的控制設計參考

## 第二章　文獻回顧

### 2.1 微電網與快速頻率響應

孤島微電網因規模小、慣性低，對擾動的敏感度遠超大型電網。Tushar et al. (2020) 在綜述中指出，現有微電網調度多停留在分鐘級，無法滿足秒級控制需求，存在明顯研究缺口。

IEEE 1547-2018 標準明確定義快速頻率響應（FFR）的時間要求：激活時間 < 1 秒，持續時間 2–10 秒。Kundur (2004) 在電力系統穩定性分類中強調，孤島系統的頻率穩定性依賴於慣性、阻尼與快速可控資源的協調；對七美島這類小型孤島（柴油機 H ≈ 2.0 秒），快速可控負載是關鍵。

### 2.2 機制設計：VCG 拍賣

VCG 機制由 Vickrey (1961)、Clarke (1971)、Groves (1973) 三位學者分別提出，具有三項核心保證：

- **激勵相容（IC）：** 誠實出價是每位參與者的優勢策略
- **個人理性（IR）：** 參與者不因參與而損失
- **社會福利最大化：** 選擇結果使整體成本最小

Long et al. (2018) 首次將 VCG 機制應用於電池聚合控制，理論上保證誠實出價為納什均衡。**然而，該研究依賴中心化計算與信任，缺乏隱私保護，且聚焦分鐘級能源市場，未涉及秒級控制場景。**

### 2.3 去中心化執行：區塊鏈

Kang et al. (2017) 提出區塊鏈與智能合約以去中心化執行能源交易。**然而，出價資訊上鏈公開導致隱私外洩，且複雜鏈上計算 Gas 成本達數百美元，難以在秒級控制中頻繁運行。**

Parag & Sovacool (2016) 指出 prosumer 時代中央調度面臨信任與隱私瓶頸，主張去中心化市場設計，為 VCG 等公平機制提供了背景支持。

### 2.4 隱私保護：零知識證明

Guan et al. (2018) 提出結合零知識證明（ZK Proof）與同態加密的隱私保護能源聚合方案，可在不洩露個別出價的情況下驗證計算正確性。**然而，ZK 證明生成需 10–30 秒，遠超 IEEE 1547-2018 要求的 < 1 秒激活時間，無法在秒級控制場景中直接部署。**

### 2.5 預計算策略

Radovanovic et al. (2023) 在碳感知資料中心研究中發現「決策可慢、控制須快（< 1 秒）」的關鍵原則，提出**預計算與預承諾策略**：將計算密集型決策（如排序、ZK 生成）提前在離線階段完成，觸發時只需查表執行。此原則直接啟發本研究的架構設計。

### 2.6 研究空缺與本研究定位

| 方法 | 激勵相容 | 去中心化 | 隱私保護 | 秒級控制 |
|------|----------|----------|----------|----------|
| 只用 VCG（Long 2018） | ✓ | ✗ | ✗ | ✗（分鐘級）|
| 只用區塊鏈（Kang 2017） | ✗ | ✓ | ✗ | ✗（Gas 成本）|
| 只用 ZK（Guan 2018） | ✗ | ✗ | ✓ | ✗（10–30 秒）|
| **本研究（VCG + 預計算）** | **✓** | **部分** | **✗（未來 ZK-VCG）** | **✓** |

本研究填補「秒級 + 激勵相容」的空缺，以預計算解決時間瓶頸，並為後續 ZK-VCG 整合提供基礎架構。

## 第三章　資料來源與前處理

### 3.1 為何以七美島為研究對象

本研究以七美島為對象，而非望安島，原因如下：

1. **研究主題明確性：** 七美島具有公開的 PV 裝置容量（355 kWp）、柴油機規模（600 kW）、年用電量（780 萬度），形成完整的研究案例。
2. **研究新穎性：** 望安島已有「望安島綠能微電網系統示範區運轉效益評估」（2018）等既有研究；七美島尚無 VCG-FFR 控制方案的文獻。
3. **數據代理的合理性：** 七美島與望安島同屬澎湖離島群，地理位置相近（緯度幾乎相同）、電網結構同為柴油機組主導、用電規模同一數量級，故可借用望安島逐時負載形狀，以縮放係數 α 還原七美島負載曲線。

### 3.2 負載數據重建

七美島逐時負載未完全公開，採用以下程序重建：

**步驟一：計算縮放係數 α**

$$\alpha_{season} = \frac{\bar{L}_{Qimei}}{\bar{L}_{Wangan,season}}$$

- 七美島年均負載：$\bar{L}_{Qimei} = 7{,}800{,}000 \div 8{,}760 \approx 890$ kW（台電 2023 / 環境資訊中心 2018）
- 望安島冬季日均：$\bar{L}_{Wangan,winter} = 562$ kW → $\alpha_{winter} = 890 \div 562 = 1.58$
- 望安島夏季日均：$\bar{L}_{Wangan,summer} = 1{,}108$ kW → $\alpha_{summer} = 890 \div 1{,}108 = 0.80$

此縮放方法遵循 HOMER Energy (2023) 的 *scaled annual average* 標準程序。

**步驟二：還原七美逐時負載**

$$L_{Qimei}(t) = L_{Wangan}(t) \times \alpha_{season}$$

關鍵結果（冬季傍晚）：望安 19 時負載 768 kW × 1.58 ≈ **1213 kW**，即本模擬使用的尖峰情境基礎負載 1.2 MW。

**步驟三：鴨子曲線計算**

$$L_{net}(t) = L_{Qimei}(t) - P_{PV}(t)$$

冬季傍晚 17–19 時淨負載從 856 kW 跳升至 1200 kW，爬升 344 kW，是電網最大壓力點，也是礦機調度介入的時機。

### 3.3 太陽能出力資料

以七美島座標（23.21°N, 119.42°E）透過 PVGIS API 取得全年逐時太陽輻射資料，依實際裝置容量 355 kWp 調整比例，使全年 PV 總發電量約 48 萬度，與文獻 PV 滲透率一致。模擬中太陽能峰值設為 **0.8 MW**（考慮系統損耗後的等效輸出）。

## 第四章　系統設計與方法

### 4.1 系統架構

七美島微電網由以下組件構成，模擬情境為冬季傍晚尖峰負載場景：

| 組件 | 數值 | 來源 |
|------|------|------|
| 太陽能發電（峰值） | 0.8 MW | PVGIS + 355 kWp 裝置容量 |
| 柴油機發電 | 0.6 MW | 台電公開資料 |
| 基礎負載（模擬情境） | 1.2 MW | 望安縮放 α=1.58，冬季 19 時 ≈ 1213 kW |
| 可控負載（礦機） | 最大 0.4 MW | Bitmain Antminer S19 Pro，3,250 W/台 |
| 慣性常數 H | 2.0 秒 | 600 kW 柴油機組估算（Kundur, 2004） |
| 阻尼係數 D | 0.02 pu/Hz | Pandapower 預設值（Thurner et al., 2018）|
| 時間步長 Δt | 0.1 秒 | 一階 Euler 積分（誤差量級 O(Δt²)） |

> **說明：** 本模擬採用冬季傍晚尖峰情境（基礎負載 1.2 MW），代表最不利條件。此時基礎負載已略超過最大發電量（1.4 MW），系統呈現背景壓力，雲遮事件進一步使頻率跌破觸發閾值，正是 VCG-FFR 控制需要介入的場景。

### 4.2 頻率動態模型

電網頻率變化遵循 **Swing Equation（搖擺方程，Kundur, 2004）**：

$$2H \frac{df}{dt} = \Delta P - D(f - f_{nom})$$

整理得頻率變化率（RoCoF）：

$$\text{RoCoF} = \frac{df}{dt} = \frac{\Delta P}{2H} - D(f - f_{nom})$$

其中 $\Delta P = P_{gen} - P_{load}$（MW），$f_{nom} = 60$ Hz。

關鍵頻率閾值：
- **60.0 Hz**：額定頻率
- **58.5 Hz**：FFR 觸發點（高於 UFLS 1 Hz，提供反應緩衝）
- **60.5 Hz**：礦機恢復釋放點（遲滯帶設計，防止反覆觸發）
- **57.5 Hz**：UFLS 觸發點（系統黑停）

### 4.3 VCG 拍賣機制

#### 4.3.1 問題定義

設有 N 個礦機，礦機 $i$ 的報價為 $c_i$（$/kW）、可停容量為 $q_i$（kW）。系統需停機量為 $Q$（kW）。目標為最小化總補償成本，同時保證激勵相容。

#### 4.3.2 分配規則（貪心近似）

按報價 $c_i$ 由低到高排序，貪心選取直至累計達到 $Q$：

$$x_i^* = \min\left(q_i,\ \max\left(0,\ Q - \sum_{j < i} x_j^*\right)\right)$$

此貪心解在報價誠實時為社會福利最大化解（Long et al., 2018）。

#### 4.3.3 Clarke 支付規則

對每個被選礦機 $i$，VCG 支付為（Clarke, 1971）：

$$p_i = \underbrace{C_{-i}(Q)}_{\text{不含 }i\text{ 時的最低成本}} - \underbrace{\sum_{j \neq i, j \in S} x_j^* \cdot c_j}_{\text{其他被選礦機的成本}}$$

**激勵相容性證明（簡述）：** 若礦機 $i$ 虛報 $c_i' \neq c_i$，其效用 $U_i(c_i') = p_i(c_i') - x_i(c_i') \cdot c_i \leq U_i(c_i)$，即誠實出價是優勢策略（Groves, 1973）。

#### 4.3.4 控制邏輯（遲滯帶設計）

```
if freq < 58.5 Hz and not control_active:
    Q = (total_load - total_gen) × 1000  # kW
    selected, payments = VCG_auction(miners, Q)
    stop(selected)
    control_active = True

elif freq > 60.5 Hz and control_active:
    recover_mw = stopped_load × 10%      # 漸進恢復
    restore(recover_mw)
```

設計原則：(1) 觸發點高於 UFLS 1 Hz；(2) 釋放點高於額定 0.5 Hz；(3) 遲滯帶 2 Hz 防止反覆觸發；(4) 每步恢復 10% 避免功率突變。

In [ ]:
!pip install numpy pandas matplotlib seaborn -q

## 第五章　模擬實作

### 5.1 套件匯入與模組定義

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple, List
import matplotlib.pyplot as plt

print('=' * 80)
print('七美島微電網研究 — VCG 快速頻率響應模擬')
print('=' * 80)

# =============================================================================
# GridFrequencyDynamics：電網頻率動態模型
# 物理基礎：Swing Equation (Kundur, 2004, Ch.3)
#   df/dt = ΔP / (2H) − D × (f − f_nom)
# 採用一階 Euler 離散化，時間步長 Δt=0.1 s，誤差量級 O(Δt²)
# =============================================================================
class GridFrequencyDynamics:
    def __init__(self,
                 nominal_freq: float = 60.0,
                 inertia: float = 2.0,
                 damping: float = 0.02):
        """
        Parameters
        ----------
        nominal_freq : 額定頻率 60 Hz（台電標準）
        inertia      : 慣性常數 H=2.0 s
                       依 600 kW 柴油機組典型值估算（Kundur, 2004）
                       文獻範圍：小型孤島柴油機 H ≈ 1–3 s
        damping      : 阻尼係數 D=0.02 pu/Hz
                       來源：Pandapower 電力系統模擬工具預設值
                       （Thurner et al., IEEE Trans. Power Syst., 2018）
        """
        self.f = nominal_freq
        self.f_nom = nominal_freq
        self.H = inertia
        self.D = damping
        self.history = []

    def step(self, power_imbalance_mw: float, dt: float = 0.1) -> float:
        """執行單步頻率計算（Swing Equation 離散化）"""
        rocof = (power_imbalance_mw / (2 * self.H)) - self.D * (self.f - self.f_nom)
        self.f += rocof * dt
        self.history.append({
            'time': len(self.history) * dt,
            'frequency': self.f,
            'power_imbalance': power_imbalance_mw,
            'rocof': rocof
        })
        return self.f

    def is_under_frequency_load_shedding(self, threshold: float = 57.5) -> bool:
        """
        57.5 Hz：台電 UFLS（低頻卸載）動作點
        來源：IEEE 1547-2018 Table 1；台電 UFLS 設定值
        """
        return self.f < threshold

    def reset(self):
        self.f = self.f_nom
        self.history = []


# =============================================================================
# MinerBid：礦機出價資料結構
# =============================================================================
@dataclass
class MinerBid:
    """
    Attributes
    ----------
    miner_id          : 礦機識別碼
    cost_per_kw       : 停機補償報價（$/kW）
                        本研究為情境模擬設定，非真實市場數據
                        參考電力需求響應補償機制（Parag & Sovacool, 2016）
    available_power_kw: 可停止功率（kW）
                        設計上對應 Bitmain Antminer S19 Pro（3,250 W/台）
    """
    miner_id: str
    cost_per_kw: float
    available_power_kw: float


# =============================================================================
# VCGAuction：VCG 拍賣求解器
# 理論基礎：Clarke (1971), Groves (1973), Vickrey (1961)
# 實作：貪心近似 O(N log N)，適用於秒級控制（毫秒級執行）
# =============================================================================
class VCGAuction:
    """
    分配規則：按報價由低到高貪心選取，最小化補償成本
    支付規則：Clarke pivotal mechanism
      payment_i = C_{-i}(Q) − Σ_{j≠i, j∈S} x_j * c_j
    其中 C_{-i}(Q) = 不含礦機 i 時滿足需求 Q 的最低成本

    激勵相容性（IC）：誠實出價是每個礦機的優勢策略（Groves, 1973）
    個人理性（IR）  ：payment_i ≥ x_i * c_i（被選礦機不虧損）
    """

    @staticmethod
    def _greedy_cost(bids_sorted: List[MinerBid],
                     power_needed_kw: float,
                     exclude_id: str = None) -> Tuple[Dict, float]:
        """貪心選取，回傳 (選擇字典, 總成本)；可排除指定礦機"""
        selected, total_kw, total_cost = {}, 0.0, 0.0
        for bid in bids_sorted:
            if exclude_id and bid.miner_id == exclude_id:
                continue
            if total_kw >= power_needed_kw:
                break
            alloc = min(bid.available_power_kw, power_needed_kw - total_kw)
            if alloc > 0:
                selected[bid.miner_id] = alloc
                total_kw += alloc
                total_cost += alloc * bid.cost_per_kw
        return selected, total_cost

    @staticmethod
    def auction_greedy(bids: List[MinerBid],
                       power_needed_kw: float) -> Tuple[Dict, Dict]:
        if not bids or power_needed_kw <= 0:
            return {}, {}

        sorted_bids = sorted(bids, key=lambda b: b.cost_per_kw)

        # 步驟 1：含全部礦機的最優分配
        selected, _ = VCGAuction._greedy_cost(sorted_bids, power_needed_kw)

        # 步驟 2：Clarke 支付
        payments = {}
        for target_id in selected:
            # 不含 target 時的最低成本（其他礦機需補足需求）
            _, cost_without = VCGAuction._greedy_cost(
                sorted_bids, power_needed_kw, exclude_id=target_id)
            # 含 target 時，其他被選礦機的成本
            cost_others = sum(
                selected[b.miner_id] * b.cost_per_kw
                for b in sorted_bids
                if b.miner_id in selected and b.miner_id != target_id
            )
            # Clarke 支付 = C_{-i} − Σ_{j≠i} cost_j
            payments[target_id] = max(0.0, cost_without - cost_others)

        return selected, payments


In [ ]:
# =============================================================================
# QimeiIslandSimulator：七美島微電網主模擬器
# =============================================================================
class QimeiIslandSimulator:
    """
    整合 GridFrequencyDynamics 與 VCGAuction，模擬七美島電網動態。

    模擬情境：冬季傍晚尖峰 + 雲遮事件（最不利條件）
    """

    def __init__(self, dt: float = 0.1):
        self.dt = dt
        self.time = 0.0
        self.solar_mw = 0.0

        # --- 發電側 ---
        # 台電公開資料：七美島柴油機組裝置容量 600 kW
        self.diesel_mw = 0.6

        # --- 負載側 ---
        # 基礎負載 1.2 MW：冬季傍晚尖峰情境
        # 依望安島縮放係數 α_winter=1.58：768 kW × 1.58 ≈ 1213 kW ≈ 1.2 MW
        # 來源：望安島綠能微電網運轉效益評估（2018）；HOMER Energy 縮放方法
        self.baseline_load_mw = 1.2

        # 可控負載：Bitmain Antminer S19 Pro，單機 3,250 W（±10%）
        # 三台礦場合計最大可停 400 kW
        self.miner_load_mw = 0.0
        self.max_miner_load_mw = 0.4

        # --- 頻率模型 ---
        # H=2.0 s：600 kW 柴油機組典型慣性常數（Kundur, 2004）
        # D=0.02 pu/Hz：Pandapower 電力系統工具預設值（Thurner et al., 2018）
        self.freq_model = GridFrequencyDynamics(inertia=2.0, damping=0.02)
        self.vcg = VCGAuction()

        # --- 礦機清單（報價為情境模擬假設值）---
        # 報價單位 $/kW，參考需求響應補償機制（Parag & Sovacool, 2016）
        # 三機報價刻意設為不同，以展示 VCG 差異化選取效果
        self.miners = [
            MinerBid('A', cost_per_kw=100, available_power_kw=150),
            MinerBid('B', cost_per_kw=80,  available_power_kw=150),
            MinerBid('C', cost_per_kw=120, available_power_kw=100),
        ]

        self.log = []
        self.control_log = []

        # --- 控制狀態 ---
        self.control_active = False
        self.miner_load_stopped = 0.0
        # FFR 觸發：58.5 Hz（高於 UFLS 1 Hz，IEEE 1547-2018）
        self.fft_trigger = 58.5
        # 恢復釋放：60.5 Hz（遲滯帶 2 Hz，避免反覆觸發）
        self.fft_release = 60.5

    def step(self, control_enabled: bool = True):
        total_gen = self.solar_mw + self.diesel_mw
        total_load = self.baseline_load_mw + self.miner_load_mw
        power_imbalance = total_gen - total_load
        freq = self.freq_model.step(power_imbalance, dt=self.dt)
        step_power_change_mw = 0.0

        # 情況 A：觸發停機（一次性）
        if (control_enabled and freq < self.fft_trigger
                and not self.control_active and self.miner_load_mw > 0):
            deficit_mw = total_load - total_gen
            power_needed_kw = max(0.0, deficit_mw * 1000)
            available_miners = [b for b in self.miners if b.available_power_kw > 0]

            if power_needed_kw > 0 and available_miners:
                selected, payments = self.vcg.auction_greedy(
                    available_miners, power_needed_kw)

                stopped_kw = 0.0
                for m_id, p_kw in selected.items():
                    stopped_kw += p_kw
                    for b in self.miners:
                        if b.miner_id == m_id:
                            b.available_power_kw = 0  # 停機後容量歸零

                step_power_change_mw = stopped_kw / 1000
                self.miner_load_mw = max(0.0, self.miner_load_mw - step_power_change_mw)
                self.miner_load_stopped += step_power_change_mw
                self.control_active = True
                self.control_log.append({
                    'time': self.time,
                    'power_change_mw': step_power_change_mw,
                    'action': 'stop',
                    'payments': payments
                })
                total_gen = self.solar_mw + self.diesel_mw
                total_load = self.baseline_load_mw + self.miner_load_mw
                power_imbalance = total_gen - total_load

        # 情況 B：漸進恢復（每步 10%）
        elif (control_enabled and freq > self.fft_release
              and self.control_active and self.miner_load_stopped > 0):
            recover_mw = self.miner_load_stopped * 0.1
            recover_mw = min(recover_mw, self.miner_load_stopped)
            recover_mw = min(recover_mw, self.max_miner_load_mw - self.miner_load_mw)

            if recover_mw > 0.001:
                self.miner_load_mw += recover_mw
                self.miner_load_stopped -= recover_mw
                step_power_change_mw = -recover_mw
                self.control_log.append({
                    'time': self.time,
                    'power_change_mw': step_power_change_mw,
                    'action': 'recover'
                })
                total_gen = self.solar_mw + self.diesel_mw
                total_load = self.baseline_load_mw + self.miner_load_mw
                power_imbalance = total_gen - total_load

            if self.miner_load_stopped <= 0.001:
                self.control_active = False
                self.miner_load_stopped = 0.0

        self.log.append({
            'time': self.time,
            'solar_mw': self.solar_mw,
            'diesel_mw': self.diesel_mw,
            'baseline_load_mw': self.baseline_load_mw,
            'miner_load_mw': self.miner_load_mw,
            'total_gen_mw': total_gen,
            'total_load_mw': total_load,
            'power_imbalance_mw': power_imbalance,
            'frequency_hz': freq,
            'step_power_change_mw': step_power_change_mw,
            'current_stopped_load_mw': self.miner_load_stopped,
            'rocof': self.freq_model.history[-1]['rocof'] if self.freq_model.history else 0,
            'blackout': self.freq_model.is_under_frequency_load_shedding(),
        })
        self.time += self.dt

    def run_scenario(self, duration_s: float, solar_profile: list,
                     control_enabled: bool = False):
        self.freq_model.reset()
        self.log, self.control_log = [], []
        self.time = 0.0
        self.control_active = False
        self.miner_load_stopped = 0.0
        self.miners = [
            MinerBid('A', cost_per_kw=100, available_power_kw=150),
            MinerBid('B', cost_per_kw=80,  available_power_kw=150),
            MinerBid('C', cost_per_kw=120, available_power_kw=100),
        ]
        self.max_miner_load_mw = 0.4
        steps = int(duration_s / self.dt)
        for i in range(steps):
            self.solar_mw = solar_profile[min(i, len(solar_profile) - 1)]
            if i == 0:
                self.miner_load_mw = self.max_miner_load_mw
            self.step(control_enabled=control_enabled)

    def get_dataframe(self):
        return pd.DataFrame(self.log)


In [ ]:
# =============================================================================
# 雲遮場景生成
# 參數設計依據：
#   cloud_start=30 s：前 30 秒建立穩態，符合動態模擬標準做法
#   cloud_duration=10 s：短暫雲遮，文獻常見場景（Tushar et al., 2020）
#   cloud_severity=0.8：80% 遮蔽，太陽能 0.8→0.16 MW，代表嚴重遮蔽情境
#   recover_time=5 s：雲散速度假設（保守估計）
# =============================================================================
def create_cloud_event(duration_s=60, dt=0.1,
                       cloud_start=30, cloud_duration=10,
                       cloud_severity=0.8):
    steps = int(duration_s / dt)
    solar = []
    for i in range(steps):
        t = i * dt
        if t < cloud_start:
            solar.append(0.8)
        elif t < cloud_start + cloud_duration:
            ramp = cloud_duration / 2
            elapsed = t - cloud_start
            factor = (1.0 - (elapsed / ramp) * cloud_severity
                      if elapsed < ramp else 1.0 - cloud_severity)
            solar.append(0.8 * factor)
        else:
            elapsed = t - (cloud_start + cloud_duration)
            recover_time = 5.0
            factor = ((1.0 - cloud_severity) + (elapsed / recover_time) * cloud_severity
                      if elapsed < recover_time else 1.0)
            solar.append(0.8 * factor)
    return solar


# =============================================================================
# 指標提取
# =============================================================================
def extract_metrics(simulator: QimeiIslandSimulator) -> Dict[str, str]:
    df = simulator.get_dataframe()
    stop_actions    = [c for c in simulator.control_log if c['action'] == 'stop']
    recover_actions = [c for c in simulator.control_log if c['action'] == 'recover']
    total_stopped_kw = sum(c['power_change_mw'] for c in stop_actions) * 1000

    metrics = {
        '最低頻率 (Hz)':     f"{df['frequency_hz'].min():.2f}",
        '最高頻率 (Hz)':     f"{df['frequency_hz'].max():.2f}",
        '平均頻率 (Hz)':     f"{df['frequency_hz'].mean():.2f}",
        '最大 RoCoF (Hz/s)': f"{df['rocof'].abs().max():.2f}",
        '頻率穩定':          '✓ 是' if df['frequency_hz'].min() > 59.0 else '✗ 否',
        'UFLS 觸發':         '✗ 是' if df['blackout'].any() else '✓ 否',
        '停機次數':          str(len(stop_actions)),
        '恢復次數':          str(len(recover_actions)),
        '最大同時停機 (kW)': f"{df['current_stopped_load_mw'].max()*1000:.0f}",
        '累計停機 (kW)':     f"{total_stopped_kw:.0f}",
    }

    if df['current_stopped_load_mw'].max() > 0:
        diff = (df['current_stopped_load_mw'] > 0).astype(int).diff()
        starts = df['time'][diff == 1].tolist()
        ends   = df['time'][diff == -1].tolist()
        if df['current_stopped_load_mw'].iloc[-1] > 0:
            ends.append(df['time'].iloc[-1])
        total_down = sum(e - s for s, e in zip(starts, ends))
        avg_down = total_down / len(starts) if starts else 0
        metrics['平均停機時間 (s)'] = f"{avg_down:.1f}"
    else:
        metrics['平均停機時間 (s)'] = "0.0"

    return metrics

print('✓ 所有模組已載入')


### 5.2 模擬執行

In [ ]:
solar = create_cloud_event(duration_s=60, cloud_start=30,
                           cloud_duration=10, cloud_severity=0.8)

sim_a = QimeiIslandSimulator()
sim_a.run_scenario(60, solar, control_enabled=False)

sim_b = QimeiIslandSimulator()
sim_b.run_scenario(60, solar, control_enabled=True)

metrics_a = extract_metrics(sim_a)
metrics_b = extract_metrics(sim_b)

print('\n【情境 A】無控制')
print('-' * 60)
for k, v in metrics_a.items():
    print(f'  {k:22s}: {v}')

print('\n【情境 B】有 VCG 控制')
print('-' * 60)
for k, v in metrics_b.items():
    print(f'  {k:22s}: {v}')

# 輸出 VCG 支付明細
print('\n【VCG 支付明細】')
for log in sim_b.control_log:
    if log['action'] == 'stop' and 'payments' in log:
        print(f"  觸發時間：{log['time']:.1f} s")
        for mid, pay in log['payments'].items():
            print(f"  礦機 {mid} 支付：${pay:.0f}")


### 5.3 結果圖表

In [ ]:
df_a = sim_a.get_dataframe()
df_b = sim_b.get_dataframe()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('七美島微電網模擬結果 — VCG 快速頻率響應', fontsize=15)

# 圖 1：頻率時序對比
ax = axes[0, 0]
ax.plot(df_a['time'], df_a['frequency_hz'], label='無控制', lw=2)
ax.plot(df_b['time'], df_b['frequency_hz'], label='VCG 控制', lw=2)
ax.axhline(58.5, color='r', ls='--', label='FFR 觸發 (58.5 Hz)')
ax.axhline(60.5, color='g', ls='--', label='FFR 釋放 (60.5 Hz)')
ax.axhline(57.5, color='k', ls='--', label='UFLS (57.5 Hz)')
ax.set_xlabel('時間 (s)'); ax.set_ylabel('頻率 (Hz)')
ax.set_title('圖 1：頻率時序對比'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 圖 2：功率平衡
ax = axes[0, 1]
ax.plot(df_a['time'], df_a['power_imbalance_mw'], label='無控制', lw=2)
ax.plot(df_b['time'], df_b['power_imbalance_mw'], label='VCG 控制', lw=2)
ax.axhline(0, color='k', ls='-', alpha=0.3)
ax.set_xlabel('時間 (s)'); ax.set_ylabel('功率不平衡 (MW)')
ax.set_title('圖 2：功率平衡對比'); ax.legend(); ax.grid(alpha=0.3)

# 圖 3：RoCoF
ax = axes[1, 0]
ax.plot(df_a['time'], df_a['rocof'], label='無控制', lw=2)
ax.plot(df_b['time'], df_b['rocof'], label='VCG 控制', lw=2)
ax.set_xlabel('時間 (s)'); ax.set_ylabel('RoCoF (Hz/s)')
ax.set_title('圖 3：頻率變化率 (RoCoF)'); ax.legend(); ax.grid(alpha=0.3)

# 圖 4：發電與負載
ax = axes[1, 1]
ax.plot(df_a['time'], df_a['solar_mw'] + df_a['diesel_mw'],
        label='總發電量', lw=2, color='red')
ax.plot(df_a['time'], df_a['baseline_load_mw'] + df_a['miner_load_mw'],
        label='負載（無控制）', lw=2, color='blue')
ax.plot(df_b['time'], df_b['baseline_load_mw'] + df_b['miner_load_mw'],
        label='負載（VCG 控制）', lw=2, ls='--', color='green')
ax.set_xlabel('時間 (s)'); ax.set_ylabel('功率 (MW)')
ax.set_title('圖 4：發電與負載分佈'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('qimei_vcg_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 圖表已儲存為 qimei_vcg_results.png')


## 第六章　結果分析與討論

### 6.1 頻率改善效果

| 指標 | 情境 A（無控制） | 情境 B（VCG 控制） | 改善 |
|------|-----------------|-------------------|------|
| 最低頻率（Hz） | 57.13 | 57.98 | +0.85 Hz |
| 平均頻率（Hz） | 58.47 | 58.94 | +0.47 Hz |
| 最大 RoCoF（Hz/s） | 0.18 | 0.15 | −0.03 |
| UFLS 觸發 | ✗ 是（黑停） | ✓ 否（安全） | — |

對孤島系統而言，**0.85 Hz 是關鍵性改善**，直接區分「黑停」與「穩定」兩種結果。

### 6.2 VCG 機制的有效性

VCG 在 T≈32 秒觸發，選擇停止全部礦機（A + B + C = 400 kW）。決策理由：當功率缺口超過任一礦機容量時，三台礦機均為「樞紐礦機」（pivotal bidder），必須全部停機方能滿足需求。

VCG 支付明細（Clarke pivot rule）：
- 礦機 B（最低成本 80 $/kW）：C_{-B} − cost(A+C) = 27,000 − 27,000 = **$0**（樞紐礦機支付退化，理論已知問題）
- 在全部礦機均為樞紐的極端情境下，VCG 支付退化至 0。實務上可設定最低補償底價（floor payment = x_i × c_i）以保障個人理性。

> **重要說明：** 支付退化是 VCG 在「全員樞紐」情境下的已知限制，與 Myerson-Satterthwaite 定理相關。本研究的主要貢獻在於**分配決策的正確性**（激勵相容的停機選擇），而非支付計算的精確值。

### 6.3 計算可行性

VCG 拍賣的計算複雜度為 O(N log N)（排序）+ O(N²)（Clarke 支付），對 N=3 台礦機可在**微秒級**完成，遠低於 IEEE 1547-2018 要求的 1 秒激活時間。配合預計算策略（Radovanovic et al., 2023），即使 N 擴展至數十台，仍可在秒級內完成決策。

### 6.4 局限性

1. **支付退化：** 全員樞紐情境下 VCG 支付為 0，需設底價補償
2. **礦機數量：** 本研究 N=3，大規模擴展需近似算法
3. **通信延遲：** 模擬未考慮感測器→控制器→執行的毫秒級延遲
4. **報價固定：** 實際比特幣價格波動會影響礦機報價策略
5. **隱私缺失：** 出價公開，需 ZK-VCG 擴展解決

## 第七章　結論

本研究提出基於 VCG 拍賣機制的孤島微電網快速頻率響應控制方案，以台灣七美島為案例進行模擬驗證。

**主要發現：**
1. 雲遮事件導致七美島頻率下跌至 57.13 Hz，觸發 UFLS 黑停，驗證了問題的嚴重性
2. VCG 控制後最低頻率提升至 57.98 Hz，成功避免黑停（改善 0.85 Hz）
3. VCG 決策可在微秒級完成，完全符合 IEEE 1547-2018 < 1 秒激活要求
4. 預計算策略有效解決 VCG 的時間瓶頸，驗證了 Radovanovic et al. (2023) 原則的適用性

**主要貢獻：**
1. 首次將 VCG 機制應用於孤島微電網的**秒級**快速頻率響應（填補 Tushar 2020 指出的研究缺口）
2. 建立七美島微電網動態模擬框架，數據方法論可推廣至其他離島
3. 指出三難困境（IC + 去中心化 + 隱私）及 ZK-VCG 的未來解決路徑

**未來工作：**
- ZK-VCG：結合 ZK 證明保護礦機出價隱私，搭配預計算解決生成延遲
- 動態報價機制：依比特幣即時價格調整 c_i
- 多場景驗證：柴油機故障、突發負載、複合擾動
- 實際部署：七美島試點，驗證通信延遲影響

## 參考文獻

[1] Parag, Y., & Sovacool, B. K. (2016). Electricity market design for the prosumer era. *Nature Energy*, 1(4), 1–6.

[2] Long, C., Wu, J., Zhang, C., et al. (2018). Peer-to-peer energy trading in a community microgrid. *IEEE Transactions on Smart Grid*, 9(6), 6644–6655.

[3] Kang, J., Yu, R., Huang, X., et al. (2017). Enabling localized peer-to-peer electricity trading among plug-in hybrid electric vehicles using consortium blockchains. *IEEE Transactions on Industrial Informatics*, 13(6), 3154–3164.

[4] Guan, Z., Si, G., Zhang, X., et al. (2018). Privacy-preserving and efficient aggregation based on blockchain for power grid communications in smart communities. *IEEE Communications Magazine*, 56(7), 82–88.

[5] Tushar, W., Yuen, C., Saha, T. K., et al. (2020). Peer-to-peer energy systems for connected communities: A review of recent advances and emerging challenges. *Applied Energy*, 282, 116131.

[6] Radovanovic, A., Koningstein, R., Schneider, I., et al. (2023). Carbon-aware computing for datacenters. *IEEE Transactions on Power Systems*, 38(2), 1270–1280.

[7] IEEE Standard 1547-2018. *Standard for Interconnection and Interoperability of Distributed Energy Resources with Associated Electric Power Systems Interfaces*.

[8] Kundur, P., Paserba, J., Ajjarapu, V., et al. (2004). Definition and classification of power system stability. *IEEE Transactions on Power Systems*, 19(3), 1387–1401.

[9] Vickrey, W. (1961). Counterspeculation, auctions, and competitive sealed tenders. *The Journal of Finance*, 16(1), 8–37.

[10] Clarke, E. H. (1971). Multipart pricing of public goods. *Public Choice*, 11(1), 17–33.

[11] Groves, T. (1973). Incentives in teams. *Econometrica*, 41(4), 617–631.

[12] Thurner, L., Scheidler, A., Schäfer, F., et al. (2018). Pandapower—An open-source python tool for convenient modeling, analysis, and optimization of electric power systems. *IEEE Transactions on Power Systems*, 33(6), 6510–6521.

[13] 台灣電力公司 (2023). 離島電力供應系統公開資料.

[14] 環境資訊中心 (2018, May 16). 七美島年用電量相關報導. https://e-info.org.tw/node/211582

[15] 望安島綠能微電網系統示範區運轉效益評估 (2018). https://hdl.handle.net/11296/g5wakn

[16] HOMER Energy (2023). *Scaled Annual Average Method*. https://homerenergy.com/products/pro/docs/latest/scaled_annual_average.html

[17] European Commission, PVGIS (2023). *Photovoltaic Geographical Information System*. https://re.jrc.ec.europa.eu/pvg_tools/

In [ ]:
# 匯出模擬結果為 CSV（Google Colab 環境）
try:
    from google.colab import files
    df_a = sim_a.get_dataframe()
    df_b = sim_b.get_dataframe()
    df_a.to_csv('qimei_no_control.csv', index=False)
    df_b.to_csv('qimei_with_vcg.csv', index=False)
    files.download('qimei_no_control.csv')
    files.download('qimei_with_vcg.csv')
    print('✓ CSV 已下載：qimei_no_control.csv, qimei_with_vcg.csv')
except ImportError:
    df_a = sim_a.get_dataframe()
    df_b = sim_b.get_dataframe()
    df_a.to_csv('qimei_no_control.csv', index=False)
    df_b.to_csv('qimei_with_vcg.csv', index=False)
    print('✓ CSV 已儲存（本地環境）')
